[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/06_rnn/06_rnn_solutions.ipynb)

# 06. RNN — 연습 문제 해설

[06_rnn.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/06_rnn/06_rnn.ipynb) 끝의 연습 문제 3개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

> **읽는 법** — 먼저 직접 풀어본 뒤 보세요. 셀은 위에서부터 순서대로 실행해야 하고,
> 실행 결과는 저장되어 있지 않으니 직접 실행해야 출력이 나타납니다.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q torch matplotlib numpy

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("device:", device)

## 연습 1. Char-RNN — hidden_size를 2~3으로 줄이면?

In [ ]:
chars = list("hielo")
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for i, c in enumerate(chars)}

text = "hihello"
x_str, y_str = text[:-1], text[1:]
x_idx = [char2idx[c] for c in x_str]
y_idx = [char2idx[c] for c in y_str]
n_classes = len(chars)

# eye(n): 단위행렬. 각 행이 곧 원-핫 벡터라 문자 인코딩에 그대로 쓴다
X = torch.tensor(np.eye(n_classes)[x_idx], dtype=torch.float32).unsqueeze(0).to(device)
Y = torch.tensor(y_idx, dtype=torch.long).unsqueeze(0).to(device)

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out)

def train_char_rnn(hidden_size, epochs=200, lr=0.1):
    torch.manual_seed(0)
    model = CharRNN(n_classes, hidden_size, n_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        optimizer.zero_grad()
        out = model(X)
        loss = loss_fn(out.view(-1, n_classes), Y.view(-1))
        loss.backward()
        optimizer.step()
    pred = out.argmax(dim=2).squeeze(0).cpu().numpy()
    pred_str = "".join(idx2char[i] for i in pred)
    return loss.item(), pred_str

for h in [2, 3, 8]:
    loss, pred_str = train_char_rnn(hidden_size=h)
    correct = "✅" if pred_str == "ihello" else "❌"
    print(f"hidden_size={h:<3} 최종 loss={loss:.4f}  예측='{pred_str}'  목표='ihello' {correct}")

**해설**
- `"hihell" -> "ihello"`를 맞히려면, RNN은 같은 입력 글자 `'h'` 뒤에도 문맥에 따라 다른 다음 글자(처음 `h` 다음엔 `i`, 이후 `h` 다음엔 여전히 `i`지만 두 번째 `l` 다음엔 `l`이 아니라 `o`)를 구분해서 예측해야 합니다. 즉 **바로 직전 글자만으로는 부족하고, 이전 문맥(은닉 상태)을 기억**해야 풉니다.
- `hidden_size`가 너무 작으면(2~3) 이 문맥 정보를 담을 "용량"이 부족해서 학습이 불안정하거나 특정 위치에서 계속 틀립니다.
- `hidden_size=8` 정도면 이 짧은 시퀀스의 문맥을 담기에 충분해서 안정적으로 `"ihello"`를 맞힙니다.
- 결론: 은닉 상태 크기는 "모델이 기억해야 하는 문맥의 복잡도"에 맞춰 정해야 합니다 — 04번 노트북의 XOR 은닉 유닛 실험과 같은 맥락(용량 부족 → 학습 실패/불안정)입니다.

## 연습 2. Sine 시계열 — RNN vs LSTM vs GRU

In [ ]:
t = np.linspace(0, 100, 1000)
series = np.sin(t)
SEQ_LEN = 20

def make_sequences(series, seq_len):
    xs, ys = [], []
    for i in range(len(series) - seq_len):
        xs.append(series[i:i + seq_len])
        ys.append(series[i + seq_len])
    return np.array(xs), np.array(ys)

X_seq, y_seq = make_sequences(series, SEQ_LEN)
split = int(len(X_seq) * 0.8)
X_train = torch.tensor(X_seq[:split], dtype=torch.float32).unsqueeze(-1).to(device)
y_train = torch.tensor(y_seq[:split], dtype=torch.float32).unsqueeze(-1).to(device)
X_test = torch.tensor(X_seq[split:], dtype=torch.float32).unsqueeze(-1).to(device)
y_test = torch.tensor(y_seq[split:], dtype=torch.float32).unsqueeze(-1).to(device)

`cell_cls`를 인자로 받게 만들어 **RNN·LSTM·GRU를 같은 구조로** 찍어낼 수 있게 했습니다.
셋 다 인터페이스가 같아서 클래스 이름만 바꿔 끼우면 됩니다.

In [ ]:
class TimeSeriesModel(nn.Module):
    def __init__(self, cell_cls, hidden_size=16):
        super().__init__()
        self.rnn = cell_cls(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

def train_and_eval(cell_cls, epochs=150, lr=0.01):
    torch.manual_seed(0)
    model = TimeSeriesModel(cell_cls).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        test_mse = loss_fn(model(X_test), y_test).item()
    return model, test_mse

# LSTM: 게이트로 긴 의존 관계를 기억 / GRU: LSTM 간소화판. 셋 다 인터페이스가 같아 바꿔 끼울 수 있다
cells = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}
models, mses = {}, {}
for name, cell_cls in cells.items():
    model, test_mse = train_and_eval(cell_cls)
    models[name] = model
    mses[name] = test_mse
    print(f"{name:<5} test MSE = {test_mse:.6f}")

세 모델의 예측을 실제값과 함께 그려 비교합니다. 범례에 test MSE도 함께 표시했습니다.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(y_test.cpu().numpy().flatten(), label="실제값", linewidth=2)
for name, model in models.items():
    with torch.no_grad():
        pred = model(X_test).cpu().numpy().flatten()
    plt.plot(pred, "--", label=f"{name} 예측 (MSE={mses[name]:.5f})")
plt.title("RNN vs LSTM vs GRU — Sine 시계열 예측")
plt.legend()
plt.show()

**해설**
- `nn.RNN`을 `nn.LSTM`/`nn.GRU`로 바꾸는 것은 인터페이스상으로는 클래스 이름만 바꾸면 될 정도로 간단합니다 (입출력 shape 동일).
- 이 sine 곡선처럼 비교적 짧고 규칙적인 시퀀스(`SEQ_LEN=20`)에서는 기본 RNN도 이미 잘 작동해서 셋의 성능 차이가 크지 않을 수 있습니다.
- LSTM/GRU는 **게이트(gate) 구조**로 어떤 정보를 기억하고 잊을지 학습하기 때문에, 시퀀스가 훨씬 길어지거나 장기 의존성(long-term dependency)이 중요한 문제일수록 기본 RNN보다 확실히 유리해집니다 — Vanishing Gradient 문제에 더 강건합니다.
- 실무에서는 특별한 이유가 없다면 기본 `nn.RNN`보다 `nn.GRU`(가볍고 성능 좋음) 또는 `nn.LSTM`(가장 널리 검증됨)을 기본 선택지로 삼는 경우가 많습니다.

## 연습 3. `SEQ_LEN`을 20에서 5로 줄이면?

`SEQ_LEN`은 **"다음 값을 맞히기 위해 과거 몇 개를 보여줄 것인가"** 입니다.
20에서 5로 줄인다는 건 모델에게 주는 단서를 4분의 1로 깎는다는 뜻입니다.

먼저 이 데이터에서 5와 20이 각각 어느 정도 길이인지부터 감을 잡아야 합니다.
`t = np.linspace(0, 100, 1000)`이므로 샘플 간격은 약 0.1rad이고, 사인 한 주기(2π)는 약 63개 샘플입니다.
즉 `SEQ_LEN=20`은 한 주기의 약 32%, `SEQ_LEN=5`는 약 8%밖에 보지 못합니다.

여러 값으로 학습해 test MSE를 비교합니다. 앞에서 만든 `make_sequences()`와
`TimeSeriesModel`을 그대로 재사용합니다.

In [ ]:
# SEQ_LEN만 바꿔가며 같은 조건으로 학습한다.
# 매번 torch.manual_seed(0)을 다시 부르는 이유: 초기 가중치가 달라지면
# SEQ_LEN 때문에 생긴 차이인지 운 때문인지 구분할 수 없다.
def run_seq_len(seq_len, epochs=150, lr=0.01):
    Xs, ys = make_sequences(series, seq_len)
    sp = int(len(Xs) * 0.8)
    Xtr = torch.tensor(Xs[:sp], dtype=torch.float32).unsqueeze(-1).to(device)
    ytr = torch.tensor(ys[:sp], dtype=torch.float32).unsqueeze(-1).to(device)
    Xte = torch.tensor(Xs[sp:], dtype=torch.float32).unsqueeze(-1).to(device)
    yte = torch.tensor(ys[sp:], dtype=torch.float32).unsqueeze(-1).to(device)

    torch.manual_seed(0)
    model = TimeSeriesModel(nn.RNN).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(Xtr), ytr)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        test_mse = loss_fn(model(Xte), yte).item()
        pred = model(Xte).cpu().numpy().flatten()
    return test_mse, pred, yte.cpu().numpy().flatten()


dt = t[1] - t[0]
period_samples = 2 * np.pi / dt
print(f"샘플 간격 {dt:.3f}rad, 사인 한 주기 = 약 {period_samples:.0f}개 샘플")
print()

results = {}
for seq_len in [5, 10, 20, 40]:
    mse, pred, actual = run_seq_len(seq_len)
    results[seq_len] = (mse, pred, actual)
    share = seq_len / period_samples * 100
    print(f"SEQ_LEN={seq_len:<3} 한 주기의 {share:4.1f}%를 봄   test MSE = {mse:.6f}")

위 숫자를 그림으로도 확인합니다. `SEQ_LEN=5`와 `SEQ_LEN=20`의 예측을 실제값과 겹쳐 그립니다.

In [ ]:
plt.figure(figsize=(10, 4))
for seq_len in [5, 20]:
    mse, pred, actual = results[seq_len]
    if seq_len == 5:
        plt.plot(actual[:200], label="실제값", linewidth=2, color="black")
    plt.plot(pred[:200], "--", label=f"SEQ_LEN={seq_len} 예측 (MSE={mse:.5f})")
plt.title("과거를 얼마나 보여주느냐에 따른 예측 차이")
plt.legend()
plt.show()

**해설**

- **결과부터**: `SEQ_LEN=20`의 test MSE가 약 `0.00009`인데 `SEQ_LEN=5`는 약 `0.00069`로,
  **7배 이상 나빠집니다.** 40으로 늘리면 조금 더 좋아지고요. 과거를 길게 볼수록 유리하다는
  경향이 숫자로 그대로 나옵니다.
- **왜 짧으면 불리한가**: 사인 곡선에서 `0.5`라는 값 하나만으로는 다음 값을 알 수 없습니다.
  올라가는 중에도 0.5를 지나고 내려가는 중에도 0.5를 지나기 때문입니다.
  **지금 주기의 어디쯤인지**를 알아야 하는데, 그 정보는 값 하나가 아니라 **최근 값들의 모양**에 들어 있습니다.
  5개는 한 주기의 8%라서 거의 직선처럼 보입니다. 기울기는 알 수 있지만 곡선이 언제 꺾일지는 알기 어렵습니다.
- **그런데 완전히 망가지지도 않습니다.** MSE 0.00069면 여전히 꽤 정확합니다.
  사인파는 노이즈가 없고 규칙이 단순해서, 짧은 창으로도 값과 기울기만 알면 어느 정도는 맞힐 수 있기 때문입니다.
  **실제 데이터라면 훨씬 심하게 무너집니다.**
- **반대로 무작정 늘리면 되는 것도 아닙니다.** 창이 길어지면 (1) 학습 데이터 개수가 줄고
  (`len(series) - seq_len`), (2) 계산이 느려지고, (3) 기본 RNN은 시점이 길어질수록
  앞쪽을 잊습니다(본문 4절의 장기 의존성 문제). 그래서 LSTM/GRU가 필요해집니다.
- **실무에서 정하는 법**: `SEQ_LEN`은 감으로 정하는 값이 아니라 **데이터의 주기에서 나옵니다.**
  일 단위 매출이면 최소 7(주 단위 패턴), 월 패턴까지 보려면 30 이상. 여기서는
  "한 주기의 몇 %를 보여주는가"가 기준이었습니다.